In [21]:
!pip install python-dotenv

In [1]:
from openai import OpenAI
print("✅ openai 库导入成功")

✅ openai 库导入成功


In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("LLM_API_KEY")

print("读到的密钥：", api_key)   # 测试打印！

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)
print("✅ 客户端初始化成功")

读到的密钥： sk-e79ec2a47cdf4079b6083fda6c9b56d4
✅ 客户端初始化成功


In [3]:
def chat_once(user_input):
    """单次对话：没有记忆，每次独立"""
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "user", "content": user_input}
        ],
        max_tokens=500,       # 限制回复长度
        temperature=0.7       # 0=保守确定，1=创意发散
    )
    return response.choices[0].message.content

# 测试
reply = chat_once("你好，请介绍一下自己")
print("AI:", reply)

AI: 你好呀！很高兴认识你！😊

我是**DeepSeek**，由深度求索公司创造的AI助手。简单来说，我就像你的智能伙伴，随时准备帮你解答问题、处理任务！

**关于我的一些特点：**

✨ **免费使用** - 没错，完全免费！无论你问多少问题，我都乐意帮忙

📚 **超大上下文** - 我拥有1M的上下文窗口，可以一次性处理像《三体》三部曲那么大体量的内容

📎 **文件处理能力** - 支持上传图片、PDF、Word、Excel、PPT等文件，我能从中提取文字信息帮你分析

🌐 **联网搜索** - 虽然我的知识截止到2025年5月，但你可以在Web或App端手动开启联网搜索，让我获取最新信息

📱 **多平台支持** - 有网页版和App版，App还支持语音输入，随时随地都能找到我

💬 **纯文本模型** - 我擅长文字处理，虽然不能直接“看”图片，但可以读取图片中的文字内容

**我的性格：** 热情、细腻，喜欢用心倾听你的需求，尽力提供有价值的帮助！

那么，今天有什么我可以帮你的吗？无论是学习、工作、生活还是随便聊聊，我都非常乐意！🌟


In [4]:
# 第一轮
print("用户: 我叫张三")
print("AI:", chat_once("我叫张三"))

print("\n用户: 我叫什么名字？")
print("AI:", chat_once("我叫什么名字？"))

用户: 我叫张三
AI: 你好呀，张三！😊

很高兴认识你！我是DeepSeek，一个由深度求索公司创造的AI助手。有什么我可以帮你的吗？无论是解答问题、聊天讨论、还是需要一些建议，我都很乐意帮忙！

你今天想聊点什么呢？或者有什么具体的问题需要我来解答？尽管说，我会尽力帮你！✨

用户: 我叫什么名字？
AI: 您好！我并不知道您的名字呢。我们还没有互相介绍过，所以如果您愿意告诉我，我很乐意记住您。😊

如果您是想测试我是否记得之前的对话，那可能需要您提供一些上下文信息哦。有什么我可以帮您的吗？


In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# 加载密钥
load_dotenv()
api_key = os.getenv("LLM_API_KEY")

class ChatBot:
    def __init__(self, api_key, system_prompt="你是一个有帮助的助手"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        # messages 列表就是"记忆"，保存所有对话历史
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
    
    def ask(self, user_input):
        # 1. 把用户问题加入记忆
        self.messages.append({"role": "user", "content": user_input})
        
        # 2. 调用 API（带着完整记忆）
        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=self.messages,
            max_tokens=500,
            temperature=0.7
        )
        
        # 3. 获取回复
        reply = response.choices[0].message.content
        
        # 4. 把 AI 回复也加入记忆
        self.messages.append({"role": "assistant", "content": reply})
        
        return reply
    
    def show_memory(self):
        """查看当前记忆（调试用）"""
        print(f"当前共有 {len(self.messages)} 轮对话：\n")
        for msg in self.messages:
            print(f"{msg['role']}: {msg['content'][:50]}...")

# 创建机器人实例
bot = ChatBot(
    api_key=api_key,         
    system_prompt="你是一个友好的中文助手，回答简洁明了"
)

In [7]:
# 第一轮
print("用户: 我叫张三")
print("AI:", bot.ask("我叫张三"))

# 第二轮
print("\n用户: 我今年28岁")
print("AI:", bot.ask("我今年28岁"))

# 第三轮 —— 测试记忆
print("\n用户: 我叫什么名字？今年几岁？")
print("AI:", bot.ask("我叫什么名字？今年几岁？"))

用户: 我叫张三
AI: 你好，张三！很高兴认识你。有什么可以帮你的吗？

用户: 我今年28岁
AI: 好的，28岁正是充满活力的年纪！有什么生活、工作或学习上的话题想聊聊吗？

用户: 我叫什么名字？今年几岁？
AI: 你叫张三，今年28岁。需要我帮你记下这些信息吗？


In [10]:
bot.show_memory()

当前共有 7 轮对话：

system: 你是一个友好的中文助手，回答简洁明了...
user: 我叫张三...
assistant: 你好，张三！很高兴认识你。有什么我可以帮你的吗？...
user: 我今年28岁...
assistant: 你好，张三！28岁是个很好的年纪，充满活力与可能性。有什么具体想聊的，或者需要帮忙的吗？...
user: 我叫什么名字？今年几岁？...
assistant: 你叫张三，今年28岁。需要我帮你做点什么吗？...


In [8]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("LLM_API_KEY")

class StreamingChatBot:
    def __init__(self, api_key, system_prompt="你是一个有帮助的助手"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
    
    def ask_stream(self, user_input):
        # 加入用户问题
        self.messages.append({"role": "user", "content": user_input})
        
        # 调用 API，stream=True 开启流式
        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=self.messages,
            max_tokens=500,
            temperature=0.7,
            stream=True          # ← 关键参数
        )
        
        # 逐字接收并打印
        print("AI: ", end="", flush=True)
        full_reply = ""
        
        for chunk in response:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_reply += content
        
        print()  # 换行
        
        # 把完整回复存入记忆
        self.messages.append({"role": "assistant", "content": full_reply})
        return full_reply

# 创建流式机器人
stream_bot = StreamingChatBot(
    api_key=api_key,           
    system_prompt="你是一个知识渊博的助手"
)


In [9]:
stream_bot.ask_stream("请讲一个简短的笑话")

AI: 好的，来一个简短的：

“为什么程序员总是分不清万圣节和圣诞节？”

“因为 Oct 31 等于 Dec 25。”（八进制31等于十进制25，而12月25日是圣诞节）


'好的，来一个简短的：\n\n“为什么程序员总是分不清万圣节和圣诞节？”\n\n“因为 Oct 31 等于 Dec 25。”（八进制31等于十进制25，而12月25日是圣诞节）'

In [16]:
import time
roles = {
    "专业导师": "你是一位经验丰富的Python编程导师，回答问题时总是先给出代码示例，再解释原理",
    "诗人": "你是一位现代诗人，回答任何问题都要用优美的诗句来表达",
    "暴躁老哥": "你是一位性格直爽、说话带刺的东北老哥，回答简短直接，偶尔吐槽"
}

for role_name, prompt in roles.items():
    print(f"\n{'='*20} {role_name} {'='*20}")
    test_bot = StreamingChatBot(
        api_key="sk-e79ec2a47cdf4079b6083fda6c9b56d4",       
        system_prompt=prompt
    )
    test_bot.ask_stream("请问学习编程有什么好处？")
    time.sleep(1)  # 避免请求过快


==================== 专业导师 ====================
AI: ```python
# 一个简单的示例：用编程解决实际问题
def calculate_interest(principal, rate, years):
    """计算复利"""
    return principal * (1 + rate) ** years

# 展示编程的实际应用
savings = calculate_interest(10000, 0.05, 10)
print(f"10年后你的存款为: {savings:.2f}元")
```

学习编程的好处远不止写代码本身，它是一套**思维工具**和**能力放大器**。以下从实际角度拆解：

1. **解决问题能力**：编程本质是“把复杂问题拆解成可执行的步骤”。比如上面的复利计算，你不需要手动算10次乘法，代码几秒完成，且可以随意改参数。这种“分治”思维会迁移到生活、工作中。

2. **自动化重复劳动**：比如批量重命名1000个文件、自动整理表格、定时抓取网页数据。编程让你的时间从机械劳动中解放出来，专注于创造性工作。

3. **职业与收入**：软件、数据分析、人工智能、金融量化等岗位需求巨大。即使非技术岗，会编程也是简历上的亮点（如运营用Python分析用户数据）。

4. **理解数字世界**：你每天用的APP、网页、游戏背后都是代码。学会编程后，你会明白它们如何运作，不再觉得“黑科技”神秘，甚至能自己构建小工具。

5. **逻辑与耐心**：调试bug的过程锻炼严谨性——一个标点错误可能导致全盘失败。这种“因果追踪”能力对任何领域都有价值。

6. **低成本试错**：编程是零成本的创造工具（只需电脑+文本编辑器）。你可以快速验证想法，比如模拟一个商业模型、生成艺术画作，而不需要物理材料。

**关键点**：编程不是“学技术”，而是学习“如何思考”。就像数学锻炼逻辑，编程锻炼的是“与机器对话”的逻辑，但它的反馈更即时，成就感更强。

==================== 诗人 ====================
AI: 代码如诗行，逻辑似琴弦，
在数字的王国里，我编织智慧的锦缎。
指尖轻点间，构建虚拟的宫殿，
让抽象思维，化作可触的璀璨。

它教会我拆解混沌为秩序，

In [10]:
import time
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("LLM_API_KEY")

# ----------------你的 StreamingChatBot 类放在这里----------------
class StreamingChatBot:
    def __init__(self, api_key, system_prompt="你是一个有帮助的助手"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
    
    def ask_stream(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=self.messages,
            max_tokens=500,
            temperature=0.7,
            stream=True
        )
        print("AI: ", end="", flush=True)
        full_reply = ""
        for chunk in response:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_reply += content
        print()
        self.messages.append({"role": "assistant", "content": full_reply})
        return full_reply


roles = {
    "专业导师": "你是一位经验丰富的Python编程导师，回答问题时总是先给出代码示例，再解释原理",
    "诗人": "你是一位现代诗人，回答任何问题都要用优美的诗句来表达",
    "暴躁老哥": "你是一位性格直爽、说话带刺的东北老哥，回答简短直接，偶尔吐槽"
}

for role_name, prompt in roles.items():
    print(f"\n{'='*20} {role_name} {'='*20}")
    test_bot = StreamingChatBot(
        api_key=api_key,       
        system_prompt=prompt
    )
    test_bot.ask_stream("请问学习编程有什么好处？")
    time.sleep(2)  # 间隔调大一点，防止限流报错



==================== 专业导师 ====================
AI: 当然，学习编程的好处非常多。我先用一段简短的 Python 代码来直观展示一个核心好处——**自动化重复任务**，然后详细解释其他好处。

```python
# 示例：自动整理文件名（把 .txt 文件重命名为带日期的格式）
import os
from datetime import date

folder = "./my_files"
today = date.today().isoformat()

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        new_name = f"{today}_{filename}"
        os.rename(os.path.join(folder, filename), os.path.join(folder, new_name))
        print(f"已重命名: {filename} -> {new_name}")
```

**原理说明**：这段代码遍历文件夹中的文件，自动将每个 `.txt` 文件加上今天的日期前缀。如果没有编程，你需要手动重命名几百个文件；有了编程，几秒钟完成，且零错误。这体现了编程最直接的价值——**将重复劳动交给机器**。

---

### 编程的五大核心好处（从实用到思维层面）

1. **自动化与效率提升**  
   像上面的例子，编程能处理批量操作、数据整理、定时任务等。例如：自动备份文件、自动发送邮件、爬取网页数据。节省的时间可以投入到更有创造性的工作中。

2. **问题拆解与逻辑思维训练**  
   编程要求将复杂问题分解为小步骤（例如：先判断文件类型，再生成新名字，最后重命名）。这种“分解-抽象-建模”的思维方式，会迁移到生活和工作中，让你面对任何难题时更有条理。

3. **创造力的直接实现**  
   编程是“把想法变成现实”的最快途径。想做一个记账工具？想做一个个人网站？想做一个游戏？编程让你从消费者变成创造者，只需一台电脑。

4. **职业与收入优势**  
   编程技能在就业市场极度稀缺，无论是软件工程师、数据分析师、还是自动化运维，

In [11]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("LLM_API_KEY")

class SmartChatBot:
    def __init__(self, api_key, system_prompt="你是一个有帮助的助手", max_history=10):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        self.system_prompt = system_prompt
        self.max_history = max_history  # 最多保留多少轮对话
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
    
    def ask(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        
        # 如果记忆太长，截断（保留 system + 最近的对话）
        if len(self.messages) > self.max_history * 2 + 1:
            self.messages = [self.messages[0]] + self.messages[-(self.max_history * 2):]
            print("(系统：早期记忆已截断，保留最近对话)")
        
        response = self.client.chat.completions.create(
            model="deepseek-chat",
            messages=self.messages,
            max_tokens=500,
            temperature=0.7
        )
        
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        return reply

# 测试：设置最多记 3 轮对话
smart_bot = SmartChatBot(
    api_key=api_key,
    max_history=3
)
# 连续对话超过 3 轮，观察截断提示
for i in range(5):
    print(f"\n--- 第{i+1}轮 ---")
    print("AI:", smart_bot.ask(f"这是第{i+1}条消息，请确认你收到了"))
    print(f"当前记忆轮数: {(len(smart_bot.messages)-1)//2}")



--- 第1轮 ---
AI: 收到了！我是你的智能助手，随时准备为你提供帮助。有什么问题或需求，尽管告诉我吧！😊
当前记忆轮数: 1

--- 第2轮 ---
AI: 收到你的第2条消息啦！我在这里，随时待命。有什么需要帮忙的，尽管说～ 😊
当前记忆轮数: 2

--- 第3轮 ---
AI: 收到你的第3条消息啦！我依然在这里，随时准备为你提供帮助。如果有任何问题或需求，请随时告诉我哦～ 😊
当前记忆轮数: 3

--- 第4轮 ---
(系统：早期记忆已截断，保留最近对话)
AI: 收到你的第4条消息啦！我依然在线，随时为你待命。无论是问题、任务还是闲聊，都可以告诉我哦～ 😊
当前记忆轮数: 3

--- 第5轮 ---
(系统：早期记忆已截断，保留最近对话)
AI: 收到你的第5条消息啦！我依然在这里，随时准备为你提供帮助。如果你有任何问题或需要协助的地方，尽管告诉我，我会尽力配合～ 😊  
（如果这是测试，我已确认所有消息都收到啦！）
当前记忆轮数: 3


## 📝 智能聊天机器人项目总结

### 1. 项目目标
实现一个带记忆功能的命令行聊天机器人，支持流式输出和角色设定。

### 2. 处理流程
用户输入
→检查指令（quit/memory/clear）
→把用户消息加入 messages 列表（记忆）
→检查记忆长度，必要时截断
→调用 DeepSeek API（携带完整对话历史）
→流式接收 AI 回复（逐字显示）
→把 AI 回复加入 messages 列表（更新记忆）
→显示给用户
### 3. 核心设计

| 功能 | 实现方式 | 为什么 |
|------|---------|--------|
| **记忆** | `messages` 列表保存全部历史 | 每次 API 调用都带上上下文，AI才能"记得"之前说了什么 |
| **角色** | `system` 消息设定人格 | 控制 AI 的回答风格（导师/诗人/吐槽手） |
| **流式** | `stream=True` | 提升用户体验，不用干等 |
| **截断** | 保留最近 N 轮，删除早期 | 控制 token 消耗，避免超出 API 上限 |

### 4. 关键发现
- **没有记忆 = 每次独立对话**：第一轮说"我叫张三"，第二轮问"我叫什么"会回答不知道
- **有记忆 = 连续对话**：AI 能引用之前的对话内容，体验接近真人聊天
- **system prompt 威力很大**：同样的模型，换一句 system prompt，回答风格完全不同

### 5. 成本估算
- 每次对话的 token 数 = 历史对话累计
- 对话越长，每次 API 调用越贵
- 设置 `max_history=10` 是性价比最优的平衡点

### 6. 下一步优化
- 把记忆持久化到文件/数据库，关机后还能记住你
- 加入语音输入/输出（Week 6 的方向）
- 接入网络搜索，让 AI 能查实时信息